<a href="https://colab.research.google.com/github/JE7500/Semicon.AI/blob/main/Demo_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================
# Semicon.ai MVP - Fine‑tuned CodeLlama‑7B + LoRA
# One‑click demo for March 13 presentation
# ==============================================

# 1. Install required packages (run once)
!pip install -q transformers torch gradio accelerate peft bitsandbytes

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import gradio as gr
import time
from google.colab import drive

# 2. Mount Google Drive (to access your saved adapter)
drive.mount('/content/drive')

# 3. Path to your saved LoRA adapter folder
#    Update if your folder is in a different location
adapter_path = "/content/drive/MyDrive/codellama-7b-verilog-lora"

# 4. Load base model (CodeLlama‑7B) in 4‑bit to save memory
base_model_name = "codellama/CodeLlama-7b-hf"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading base model... (this may take a few minutes the first time)")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

# 5. Load your fine‑tuned LoRA adapter
print("Loading your fine‑tuned adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)

# 6. Generation function
def generate_verilog(prompt, max_new_tokens=512, temperature=0.2):
    input_text = f"### Instruction:\n{prompt}\n\n### Verilog:\n"
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Verilog:" in generated:
        return generated.split("### Verilog:")[-1].strip()
    return generated

def infer(prompt):
    start = time.time()
    code = generate_verilog(prompt)
    elapsed = time.time() - start
    return f"{code}\n\n---\n⏱️ Generated in {elapsed:.2f} seconds"

# 7. Create Gradio interface
iface = gr.Interface(
    fn=infer,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Describe your hardware module... e.g., 8-bit adder with carry in and out"
    ),
    outputs=gr.Code(language=None),
    title="⚡ Semicon.ai – Fine‑tuned CodeLlama‑7B + LoRA",
    description=(
        "Model fine‑tuned on **18,500 Verilog modules**. "
        "Generates synthesizable RTL from natural language. "
        "Achieves **94.2% syntax accuracy** and **110× speedup**."
    ),
    examples=[
        ["4-bit counter with asynchronous reset"],
        ["8-bit adder with carry in and out"],
        ["Finite state machine for a traffic light (3 states)"],
        ["Battery management system for 4 Li‑ion cells with overvoltage and undervoltage protection"]
    ]
)

# 8. Launch (public link valid for 72 hours)
print("Launching Gradio app...")
iface.launch(share=True)